In [1]:
import sys
import os

import pandas as pd
import csv
from transformers import AutoTokenizer, AutoModelForSequenceClassification, \
    Trainer, set_seed, TrainingArguments
from datasets import Dataset
import torch
from sklearn.utils import compute_class_weight
import numpy as np
from tqdm import tqdm
import json
import datasets
from sklearn.model_selection import train_test_split
from evaluate import load
import numpy as np

c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_dataset(file_path, language='en', dataset_name='MultiPICo', limit=None):
    """
    Carica il dataset dal unified corpus, mantenendo gli unique_id anche negli split.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            unified_corpus = json.load(f)
        print(f"Loaded unified corpus with {len(unified_corpus)} entries")
        
        df = pd.DataFrame(unified_corpus)
        
        # Filtra per lingua e dataset
        df_filtered = df[(df['language'] == language) & (df['dataset_name'] == dataset_name)]
        print(f"Filtered {dataset_name} {language.upper()} entries: {len(df_filtered)}")
        
        # Limita il numero di entries se necessario
        if limit is not None:
            df_filtered = df_filtered.head(limit)
            print(f"Using {len(df_filtered)} entries (limited)")

        # Seleziona solo le colonne utili
        df_filtered = df_filtered[['unique_id', 'post', 'annotations']].copy()
        df_filtered = df_filtered.rename(columns={'post': 'text'})
        df_filtered['label'] = df_filtered['annotations'].apply(
            lambda x: x.get('irony_appropriateness_label', None)
        )
        df_filtered = df_filtered.drop(columns=['annotations'])

        print(f"Entries with label 1: {len(df_filtered[df_filtered['label'] == 1])}")
        print(f"Entries with label 0: {len(df_filtered[df_filtered['label'] == 0])}")

        # Includiamo anche gli ID nello split
        train_df, temp_df = train_test_split(
            df_filtered,
            test_size=0.3,
            random_state=42,
            stratify=df_filtered['label']
        )

        val_df, test_df = train_test_split(
            temp_df,
            test_size=0.5,
            random_state=42,
            stratify=temp_df['label']
        )

        # Mostra distribuzioni
        for name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
            print(f'{name} len: {len(split_df)} | label 1: {len(split_df[split_df["label"]==1])} | label 0: {len(split_df[split_df["label"]==0])}')

        dataset_splits = {
            'dataset_name': dataset_name,
            'dataset_language': language,
            'training_set': train_df.reset_index(drop=True),
            'validation_set': val_df.reset_index(drop=True),
            'test_set': test_df.reset_index(drop=True)
        }

        return dataset_splits
        
    except Exception as e:
        print(f"Error loading unified corpus: {e}")
        raise e

data_path = "../1_datasets_merge/output/unified_corpus.json"

# Carica e filtra il dataset
dataset_splits = load_dataset(
    file_path=data_path,
    language='en',
    dataset_name='MultiPICo',
    limit=None  # None per usare tutti i dati, oppure un numero specifico per test
)

Loaded unified corpus with 25814 entries
Filtered MultiPICo EN entries: 2999
Entries with label 1: 882
Entries with label 0: 2117
train len: 2099 | label 1: 617 | label 0: 1482
val len: 450 | label 1: 133 | label 0: 317
test len: 450 | label 1: 132 | label 0: 318


In [8]:
def tokenize(x, tokenizer):
    return tokenizer(
        x["text"], 
        padding="max_length", #config.padding,
        truncation=True, #config.truncation,
        max_length=512, #config.max_length,
    )

def generate_data(split):
    tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base", trust_remote_code=True)

    ids, texts, labels = [], [], []

    for _, row in split.iterrows():
        ids.append(row['unique_id'])
        texts.append(row['text'])
        labels.append(row['label'])

    df = pd.DataFrame({"text":texts, "labels":labels})
    dt = Dataset.from_pandas(df)
    tokenized_dataset = dt.map(lambda x: tokenize(x, tokenizer), remove_columns=['text'])
    tokenized_dataset = tokenized_dataset.with_format("torch")
    return tokenized_dataset, ids


train = generate_data(dataset_splits['training_set'])[0] # tokenized dataset
train

Map: 100%|██████████| 2099/2099 [00:00<00:00, 3362.51 examples/s]


Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 2099
})

In [10]:
train['labels']

Column([tensor(0), tensor(0), tensor(0), tensor(1), tensor(1)])

In [15]:
labels_array = np.array(train["labels"]).astype(int)  # converte la lista in array NumPy

class_weights = compute_class_weight(
    "balanced",
    classes=np.unique(labels_array),
    y=labels_array
).astype("float32")

class_weights

array([0.70816463, 1.7009724 ], dtype=float32)

In [13]:
class_weights = compute_class_weight(
    "balanced",
    classes=np.unique(train["labels"].numpy().astype(int)),
    y=train["labels"].numpy().astype(int)).astype("float32")

AttributeError: 'Column' object has no attribute 'numpy'